In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

NOTEBOOK 03: CHUNKING STRATEGIES FOR LEGAL DOCUMENTS
====================================================

Workshop: Knowledge Management dengan LLM untuk OMBUDSMAN RI

Durasi: 45 menit (11.15-12.00)

oleh: Assoc.Prof.Dr.Eng. Farrikh Alzami, M.Kom

Tujuan:
- Understand WHY chunking strategy matters untuk RAG quality
- Compare different chunking approaches
- Implement legal document-aware chunking
- Optimize chunk size untuk Indonesian legal documents
- Evaluate chunking quality

# SECTION 1: SETUP & INSTALASI

In [ ]:
import sys
print("Installing required packages...")

requirements = [
    "sentence-transformers",
    "nltk"
]

for package in requirements:
    print(f"Installing {package}...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

print("\nDownloading NLTK data...")
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

print("\nInstalasi selesai!")


# SECTION 2: IMPORT LIBRARIES

In [ ]:
import re
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict, Tuple
from dataclasses import dataclass
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

# NLTK for sentence splitting
from nltk.tokenize import sent_tokenize

# Create output directories
os.makedirs('output_tables', exist_ok=True)
os.makedirs('output_figures', exist_ok=True)

# Visualization settings
plt.style.use('default')
sns.set_palette("husl")

print("Libraries loaded successfully!")

# Load embedding model (reuse dari Notebook 01)
print("\nLoading embedding model...")
model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
print(f"Model loaded: {model.max_seq_length} max tokens")


# SECTION 2B: FILE LOADING UTILITIES

In [ ]:
def load_document_from_file(filepath: str) -> Dict:
    """
    Load document dari .txt file
    
    File format:
    TITLE: [title]
    CATEGORY: [category]
    QUALITY: [quality]
    DOC_ID: [doc_id]
    ====...====
    
    [content]
    
    Returns:
        Dict dengan keys: title, category, quality, doc_id, content, filepath
    """
    with open(filepath, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    
    # Parse metadata (first 4 lines)
    metadata = {}
    for i in range(min(4, len(lines))):
        line = lines[i].strip()
        if ':' in line:
            key, value = line.split(':', 1)
            metadata[key.strip().lower()] = value.strip()
    
    # Find separator line (====)
    separator_idx = 0
    for i, line in enumerate(lines):
        if line.strip().startswith('==='):
            separator_idx = i
            break
    
    # Content is everything after separator
    content = ''.join(lines[separator_idx + 2:])  # Skip separator and empty line
    
    return {
        'title': metadata.get('title', 'Unknown'),
        'category': metadata.get('category', 'Unknown'),
        'quality': metadata.get('quality', 'Unknown'),
        'doc_id': metadata.get('doc_id', 'Unknown'),
        'content': content.strip(),
        'filepath': filepath
    }


def load_documents_from_folder(folder_path: str, 
                               category_filter: str = None,
                               quality_filter: str = None) -> List[Dict]:
    """
    Load semua documents dari folder
    
    Args:
        folder_path: Path ke folder containing .txt files
        category_filter: Optional filter by category (e.g., 'Peraturan')
        quality_filter: Optional filter by quality (e.g., 'HIGH')
    
    Returns:
        List of document dicts
    """
    documents = []
    
    if not os.path.exists(folder_path):
        print(f"ERROR: Folder not found: {folder_path}")
        return documents
    
    # Get all .txt files
    txt_files = [f for f in os.listdir(folder_path) if f.endswith('.txt')]
    
    print(f"Found {len(txt_files)} .txt files in {folder_path}")
    
    for filename in sorted(txt_files):
        filepath = os.path.join(folder_path, filename)
        
        try:
            doc = load_document_from_file(filepath)
            
            # Apply filters
            if category_filter and doc['category'] != category_filter:
                continue
            
            if quality_filter and doc['quality'] != quality_filter:
                continue
            
            documents.append(doc)
            
        except Exception as e:
            print(f"Warning: Could not load {filename}: {e}")
    
    return documents

print("File loading utilities defined!")


# SECTION 2C: LOAD SAMPLE LEGAL DOCUMENT

In [ ]:
print("\n" + "="*80)
print("LOADING SAMPLE LEGAL DOCUMENT FROM FILE")
print("="*80 + "\n")

# Try common paths
possible_paths = [
    '/kaggle/input/ombudsman-synthetic-docs/ombudsman-synthetic-docs/',
]

DOCS_FOLDER = None
for path in possible_paths:
    if os.path.exists(path):
        DOCS_FOLDER = path
        break

if DOCS_FOLDER is None:
    print("ERROR: Could not find synthetic_docs_txt folder!")
    print("Please update path atau upload folder.")
    print("\nSearched paths:")
    for p in possible_paths:
        print(f"  - {p}")
    sample_legal_document = ""
    sample_doc = None
else:
    print(f"Documents folder found: {DOCS_FOLDER}")
    
    # Load HIGH quality Peraturan untuk demo
    peraturan_docs = load_documents_from_folder(
        DOCS_FOLDER,
        category_filter="Peraturan",
        quality_filter="HIGH"
    )
    
    if peraturan_docs:
        # Take first HIGH quality Peraturan
        sample_doc = peraturan_docs[0]
        sample_legal_document = sample_doc['content']
        
        print(f"\nLoaded: {sample_doc['doc_id']}")
        print(f"File: {os.path.basename(sample_doc['filepath'])}")
        print(f"Title: {sample_doc['title']}")
        print(f"Category: {sample_doc['category']}")
        print(f"Quality: {sample_doc['quality']}")
        print(f"\nDocument length: {len(sample_legal_document)} characters")
        print(f"Word count: {len(sample_legal_document.split())} words")
    else:
        print("\nERROR: No HIGH quality Peraturan found in folder")
        print("Please check folder contents")
        sample_legal_document = ""
        sample_doc = None


# SECTION 3: CHUNK DATA STRUCTURE

In [ ]:
@dataclass
class Chunk:
    """
    Represent a single chunk of text with metadata
    """
    chunk_id: str
    text: str
    source_document: str
    start_position: int
    end_position: int
    metadata: Dict
    
    def __len__(self):
        return len(self.text)
    
    def word_count(self):
        return len(self.text.split())
    
    def __repr__(self):
        preview = self.text[:50] + "..." if len(self.text) > 50 else self.text
        return f"Chunk(id={self.chunk_id}, words={self.word_count()}, text='{preview}')"

print("\nChunk data structure defined!")



# SECTION 4: CHUNKING STRATEGY 1 - FIXED SIZE

Strategy 1: Fixed-Size Chunking
--------------------------------
Paling simple: potong setiap N karakter atau N kata

Pros:
- Simple, mudah implement
- Predictable chunk sizes

Cons:
- Tidak respect document structure
- Bisa potong di tengah kalimat/paragraf
- Context loss

In [ ]:
class FixedSizeChunker:
    """
    Chunking dengan fixed size (character atau word based)
    """
    
    def __init__(self, chunk_size: int = 500, overlap: int = 50, mode: str = 'char'):
        """
        Args:
            chunk_size: Size per chunk (characters atau words)
            overlap: Overlap antar chunks untuk preserve context
            mode: 'char' atau 'word'
        """
        self.chunk_size = chunk_size
        self.overlap = overlap
        self.mode = mode
    
    def chunk(self, text: str, doc_id: str = "doc") -> List[Chunk]:
        """
        Split text into fixed-size chunks
        """
        chunks = []
        
        if self.mode == 'char':
            # Character-based chunking
            start = 0
            chunk_num = 0
            
            while start < len(text):
                end = start + self.chunk_size
                chunk_text = text[start:end]
                
                chunk = Chunk(
                    chunk_id=f"{doc_id}_chunk_{chunk_num:03d}",
                    text=chunk_text,
                    source_document=doc_id,
                    start_position=start,
                    end_position=end,
                    metadata={'method': 'fixed_char', 'chunk_size': self.chunk_size}
                )
                chunks.append(chunk)
                
                start += (self.chunk_size - self.overlap)
                chunk_num += 1
        
        elif self.mode == 'word':
            # Word-based chunking
            words = text.split()
            start = 0
            chunk_num = 0
            
            while start < len(words):
                end = start + self.chunk_size
                chunk_words = words[start:end]
                chunk_text = ' '.join(chunk_words)
                
                chunk = Chunk(
                    chunk_id=f"{doc_id}_chunk_{chunk_num:03d}",
                    text=chunk_text,
                    source_document=doc_id,
                    start_position=start,
                    end_position=end,
                    metadata={'method': 'fixed_word', 'chunk_size': self.chunk_size}
                )
                chunks.append(chunk)
                
                start += (self.chunk_size - self.overlap)
                chunk_num += 1
        
        return chunks

# Demo Fixed-Size Chunking
print("\n" + "="*80)
print("STRATEGY 1: FIXED-SIZE CHUNKING")
print("="*80 + "\n")

if sample_legal_document:
    doc_id = sample_doc['doc_id'] if sample_doc else "document"
    
    fixed_chunker = FixedSizeChunker(chunk_size=300, overlap=50, mode='char')
    fixed_chunks = fixed_chunker.chunk(sample_legal_document, doc_id=doc_id)
    
    print(f"Generated {len(fixed_chunks)} chunks\n")
    print("Sample chunks:")
    print("-"*80)
    for i, chunk in enumerate(fixed_chunks[:3]):
        print(f"\nChunk {i+1}:")
        print(f"  ID: {chunk.chunk_id}")
        print(f"  Length: {len(chunk)} characters")
        print(f"  Words: {chunk.word_count()}")
        print(f"  Preview: {chunk.text[:100]}...")
else:
    print("ERROR: No document loaded. Cannot demo chunking.")
    fixed_chunks = []


# SECTION 5: CHUNKING STRATEGY 2 - SENTENCE-BASED

Strategy 2: Sentence-Based Chunking
------------------------------------
Respect sentence boundaries

Pros:
- Tidak potong di tengah kalimat
- Lebih coherent chunks
- Better context preservation

Cons:
- Variable chunk sizes
- Masih tidak respect document structure (Pasal, Ayat)


In [ ]:
class SentenceBasedChunker:
    """
    Chunking berdasarkan sentence boundaries
    """
    
    def __init__(self, max_sentences: int = 5):
        """
        Args:
            max_sentences: Maximum sentences per chunk
        """
        self.max_sentences = max_sentences
    
    def chunk(self, text: str, doc_id: str = "doc") -> List[Chunk]:
        """
        Split text into sentence-based chunks
        """
        # Tokenize into sentences (use English tokenizer for Indonesian)
        sentences = sent_tokenize(text, language='english')
        
        chunks = []
        current_chunk_sentences = []
        chunk_num = 0
        start_pos = 0
        
        for sentence in sentences:
            current_chunk_sentences.append(sentence)
            
            if len(current_chunk_sentences) >= self.max_sentences:
                chunk_text = ' '.join(current_chunk_sentences)
                
                chunk = Chunk(
                    chunk_id=f"{doc_id}_sent_{chunk_num:03d}",
                    text=chunk_text,
                    source_document=doc_id,
                    start_position=start_pos,
                    end_position=start_pos + len(chunk_text),
                    metadata={'method': 'sentence', 'num_sentences': len(current_chunk_sentences)}
                )
                chunks.append(chunk)
                
                start_pos += len(chunk_text)
                current_chunk_sentences = []
                chunk_num += 1
        
        # Handle remaining sentences
        if current_chunk_sentences:
            chunk_text = ' '.join(current_chunk_sentences)
            chunk = Chunk(
                chunk_id=f"{doc_id}_sent_{chunk_num:03d}",
                text=chunk_text,
                source_document=doc_id,
                start_position=start_pos,
                end_position=start_pos + len(chunk_text),
                metadata={'method': 'sentence', 'num_sentences': len(current_chunk_sentences)}
            )
            chunks.append(chunk)
        
        return chunks

# Demo Sentence-Based Chunking
print("\n" + "="*80)
print("STRATEGY 2: SENTENCE-BASED CHUNKING")
print("="*80 + "\n")

if sample_legal_document:
    sentence_chunker = SentenceBasedChunker(max_sentences=3)
    sentence_chunks = sentence_chunker.chunk(sample_legal_document, doc_id=doc_id)
    
    print(f"Generated {len(sentence_chunks)} chunks\n")
    print("Sample chunks:")
    print("-"*80)
    for i, chunk in enumerate(sentence_chunks[:3]):
        print(f"\nChunk {i+1}:")
        print(f"  ID: {chunk.chunk_id}")
        print(f"  Sentences: {chunk.metadata['num_sentences']}")
        print(f"  Length: {len(chunk)} characters")
        print(f"  Preview: {chunk.text[:150]}...")
else:
    print("ERROR: No document loaded. Cannot demo chunking.")
    sentence_chunks = []


# SECTION 6: CHUNKING STRATEGY 3 - LEGAL DOCUMENT AWARE

Strategy 3: Legal Document-Aware Chunking
------------------------------------------
Respect legal document structure: BAB, Pasal, Ayat, Huruf

Pros:
- Preserve semantic units (1 Pasal = 1 konsep)
- Maintain cross-references
- Best context preservation
- Domain-specific optimization

Cons:
- More complex implementation
- Requires pattern recognition
- Variable chunk sizes


In [ ]:
class LegalDocumentChunker:
    """
    Chunking yang aware terhadap struktur dokumen legal Indonesia
    
    Hierarchy:
    - BAB (Chapter)
    - Pasal (Article)
    - Ayat (Verse/Paragraph)
    - Huruf (Letter/Point)
    """
    
    def __init__(self, chunk_level: str = 'pasal'):
        """
        Args:
            chunk_level: Level chunking ('bab', 'pasal', 'ayat')
        """
        self.chunk_level = chunk_level
        
        # Regex patterns untuk detect struktur
        self.patterns = {
            'bab': re.compile(r'BAB\s+[IVX]+\s*\n', re.MULTILINE),
            'pasal': re.compile(r'Pasal\s+\d+', re.MULTILINE),
            'ayat': re.compile(r'\(\d+\)', re.MULTILINE),
            'huruf': re.compile(r'^[a-z]\.\s+', re.MULTILINE)
        }
    
    def chunk(self, text: str, doc_id: str = "doc") -> List[Chunk]:
        """
        Split text berdasarkan legal document structure
        """
        chunks = []
        
        if self.chunk_level == 'pasal':
            # Split by Pasal
            pasal_matches = list(self.patterns['pasal'].finditer(text))
            
            for i, match in enumerate(pasal_matches):
                start = match.start()
                
                # Find end (start of next Pasal or end of document)
                if i < len(pasal_matches) - 1:
                    end = pasal_matches[i + 1].start()
                else:
                    end = len(text)
                
                chunk_text = text[start:end].strip()
                
                # Extract Pasal number
                pasal_num = match.group().strip()
                
                chunk = Chunk(
                    chunk_id=f"{doc_id}_{pasal_num.replace(' ', '_')}",
                    text=chunk_text,
                    source_document=doc_id,
                    start_position=start,
                    end_position=end,
                    metadata={
                        'method': 'legal_pasal',
                        'pasal': pasal_num,
                        'level': 'pasal'
                    }
                )
                chunks.append(chunk)
        
        elif self.chunk_level == 'bab':
            # Split by BAB
            bab_matches = list(self.patterns['bab'].finditer(text))
            
            for i, match in enumerate(bab_matches):
                start = match.start()
                
                if i < len(bab_matches) - 1:
                    end = bab_matches[i + 1].start()
                else:
                    end = len(text)
                
                chunk_text = text[start:end].strip()
                
                # Extract BAB number
                bab_match = re.search(r'BAB\s+([IVX]+)', chunk_text)
                bab_num = bab_match.group(1) if bab_match else f"BAB_{i}"
                
                chunk = Chunk(
                    chunk_id=f"{doc_id}_BAB_{bab_num}",
                    text=chunk_text,
                    source_document=doc_id,
                    start_position=start,
                    end_position=end,
                    metadata={
                        'method': 'legal_bab',
                        'bab': f"BAB {bab_num}",
                        'level': 'bab'
                    }
                )
                chunks.append(chunk)
        
        return chunks

# Demo Legal Document-Aware Chunking
print("\n" + "="*80)
print("STRATEGY 3: LEGAL DOCUMENT-AWARE CHUNKING")
print("="*80 + "\n")

if sample_legal_document:
    legal_chunker = LegalDocumentChunker(chunk_level='pasal')
    legal_chunks = legal_chunker.chunk(sample_legal_document, doc_id=doc_id)
    
    print(f"Generated {len(legal_chunks)} chunks (by Pasal)\n")
    print("Sample chunks:")
    print("-"*80)
    for i, chunk in enumerate(legal_chunks[:3]):
        print(f"\nChunk {i+1}:")
        print(f"  ID: {chunk.chunk_id}")
        print(f"  Pasal: {chunk.metadata['pasal']}")
        print(f"  Length: {len(chunk)} characters")
        print(f"  Words: {chunk.word_count()}")
        print(f"  Preview: {chunk.text[:150]}...")
else:
    print("ERROR: No document loaded. Cannot demo chunking.")
    legal_chunks = []


# SECTION 7: COMPARISON - CHUNKING STRATEGIES

In [ ]:
print("\n" + "="*80)
print("COMPARISON: CHUNKING STRATEGIES")
print("="*80 + "\n")

if sample_legal_document and fixed_chunks and sentence_chunks and legal_chunks:
    # Collect statistics
    strategies_stats = {
        'Fixed-Size (300 char)': {
            'chunks': fixed_chunks,
            'count': len(fixed_chunks),
            'avg_length': np.mean([len(c) for c in fixed_chunks]),
            'std_length': np.std([len(c) for c in fixed_chunks]),
            'avg_words': np.mean([c.word_count() for c in fixed_chunks]),
            'std_words': np.std([c.word_count() for c in fixed_chunks])
        },
        'Sentence-Based (3 sent)': {
            'chunks': sentence_chunks,
            'count': len(sentence_chunks),
            'avg_length': np.mean([len(c) for c in sentence_chunks]),
            'std_length': np.std([len(c) for c in sentence_chunks]),
            'avg_words': np.mean([c.word_count() for c in sentence_chunks]),
            'std_words': np.std([c.word_count() for c in sentence_chunks])
        },
        'Legal-Aware (Pasal)': {
            'chunks': legal_chunks,
            'count': len(legal_chunks),
            'avg_length': np.mean([len(c) for c in legal_chunks]),
            'std_length': np.std([len(c) for c in legal_chunks]),
            'avg_words': np.mean([c.word_count() for c in legal_chunks]),
            'std_words': np.std([c.word_count() for c in legal_chunks])
        }
    }
    
    # Create comparison DataFrame
    comparison_data = []
    for strategy, stats in strategies_stats.items():
        comparison_data.append({
            'Strategy': strategy,
            'Num Chunks': stats['count'],
            'Avg Chars': f"{stats['avg_length']:.0f}",
            'Std Chars': f"{stats['std_length']:.0f}",
            'Avg Words': f"{stats['avg_words']:.0f}",
            'Std Words': f"{stats['std_words']:.0f}"
        })
    
    comparison_df = pd.DataFrame(comparison_data)
    print(comparison_df.to_string(index=False))
    
    # Save to CSV
    csv_path = 'output_tables/chunking_comparison.csv'
    comparison_df.to_csv(csv_path, index=False)
    print(f"\nTable saved to: {csv_path}")
    
    # Visualization
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Plot 1: Number of chunks
    ax1 = axes[0]
    strategies = list(strategies_stats.keys())
    counts = [strategies_stats[s]['count'] for s in strategies]
    colors = ['#3498db', '#e74c3c', '#2ecc71']
    
    bars = ax1.bar(strategies, counts, color=colors, alpha=0.7, edgecolor='black')
    ax1.set_ylabel('Number of Chunks', fontweight='bold')
    ax1.set_title('Chunking Strategy: Number of Chunks Generated', fontweight='bold')
    ax1.grid(axis='y', alpha=0.3)
    
    for bar, count in zip(bars, counts):
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height,
                 f'{int(count)}',
                 ha='center', va='bottom', fontweight='bold')
    
    # Plot 2: Chunk size distribution
    ax2 = axes[1]
    positions = range(len(strategies))
    avg_words = [strategies_stats[s]['avg_words'] for s in strategies]
    std_words = [strategies_stats[s]['std_words'] for s in strategies]
    
    bars = ax2.bar(positions, avg_words, yerr=std_words, color=colors, alpha=0.7,
                   capsize=5, edgecolor='black')
    ax2.set_xticks(positions)
    ax2.set_xticklabels(strategies, rotation=15, ha='right')
    ax2.set_ylabel('Average Words per Chunk', fontweight='bold')
    ax2.set_title('Chunk Size Distribution (Mean +/- Std)', fontweight='bold')
    ax2.grid(axis='y', alpha=0.3)
    
    for bar, avg in zip(bars, avg_words):
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height,
                 f'{avg:.0f}',
                 ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    
    # Save figure
    fig_path = 'output_figures/chunking_comparison.png'
    plt.savefig(fig_path, dpi=300, bbox_inches='tight')
    print(f"Figure saved to: {fig_path}")
    
    plt.show()
else:
    print("ERROR: Cannot perform comparison - missing chunks data")
    strategies_stats = {}


# SECTION 8: SEMANTIC COHERENCE EVALUATION

Evaluate chunking quality: Semantic Coherence
----------------------------------------------
Good chunks should:
1. Be internally coherent (high intra-chunk similarity)
2. Be distinct from other chunks (low inter-chunk similarity)

Metric: Coherence Score = Avg(intra-chunk similarity) - Avg(inter-chunk similarity)
Higher score = better chunking

In [ ]:
print("\n" + "="*80)
print("SEMANTIC COHERENCE EVALUATION")
print("="*80 + "\n")

def calculate_coherence_score(chunks: List[Chunk], model) -> Dict:
    """
    Calculate semantic coherence untuk chunking strategy
    """
    if not chunks:
        return {
            'coherence_score': 0.0,
            'size_consistency': 0.0,
            'avg_inter_chunk_sim': 0.0,
            'num_chunks': 0
        }
    
    # Get embeddings untuk setiap chunk
    chunk_texts = [c.text for c in chunks]
    embeddings = model.encode(chunk_texts, show_progress_bar=False)
    
    # Calculate pairwise similarities
    similarity_matrix = cosine_similarity(embeddings)
    
    # Inter-chunk similarity: average similarity dengan chunks lain
    inter_chunk_sims = []
    for i in range(len(chunks)):
        other_sims = [similarity_matrix[i][j] for j in range(len(chunks)) if i != j]
        if other_sims:
            inter_chunk_sims.append(np.mean(other_sims))
    
    avg_inter_chunk = np.mean(inter_chunk_sims) if inter_chunk_sims else 0.0
    
    # Intra-chunk: for simplification, measure consistency via std of chunk sizes
    chunk_sizes = [c.word_count() for c in chunks]
    size_consistency = 1.0 - min(np.std(chunk_sizes) / (np.mean(chunk_sizes) + 1), 1.0)
    
    # Overall coherence score
    coherence_score = size_consistency - (avg_inter_chunk * 0.5)
    
    return {
        'coherence_score': coherence_score,
        'size_consistency': size_consistency,
        'avg_inter_chunk_sim': avg_inter_chunk,
        'num_chunks': len(chunks)
    }

if strategies_stats:
    print("Calculating semantic coherence for each strategy...")
    print("This may take a minute...\n")
    
    coherence_results = {}
    for strategy, stats in strategies_stats.items():
        print(f"Evaluating: {strategy}...")
        coherence = calculate_coherence_score(stats['chunks'], model)
        coherence_results[strategy] = coherence
    
    # Display results
    coherence_data = []
    for strategy, results in coherence_results.items():
        coherence_data.append({
            'Strategy': strategy,
            'Coherence Score': f"{results['coherence_score']:.4f}",
            'Size Consistency': f"{results['size_consistency']:.4f}",
            'Avg Inter-Chunk Sim': f"{results['avg_inter_chunk_sim']:.4f}"
        })
    
    coherence_df = pd.DataFrame(coherence_data)
    print("\n" + "="*80)
    print("COHERENCE EVALUATION RESULTS")
    print("="*80 + "\n")
    print(coherence_df.to_string(index=False))
    
    # Save to CSV
    coherence_csv_path = 'output_tables/coherence_evaluation.csv'
    coherence_df.to_csv(coherence_csv_path, index=False)
    print(f"\nTable saved to: {coherence_csv_path}")
    
    # Visualization
    fig, ax = plt.subplots(figsize=(10, 6))
    
    strategies_list = list(coherence_results.keys())
    coherence_scores = [coherence_results[s]['coherence_score'] for s in strategies_list]
    colors = ['#3498db', '#e74c3c', '#2ecc71']
    
    bars = ax.barh(strategies_list, coherence_scores, color=colors, alpha=0.7, edgecolor='black')
    ax.set_xlabel('Coherence Score (Higher = Better)', fontweight='bold')
    ax.set_title('Semantic Coherence Comparison Across Chunking Strategies', fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    
    for bar, score in zip(bars, coherence_scores):
        width = bar.get_width()
        ax.text(width + 0.01, bar.get_y() + bar.get_height()/2,
                f'{score:.4f}',
                ha='left', va='center', fontweight='bold')
    
    plt.tight_layout()
    
    # Save figure
    coherence_fig_path = 'output_figures/coherence_evaluation.png'
    plt.savefig(coherence_fig_path, dpi=300, bbox_inches='tight')
    print(f"Figure saved to: {coherence_fig_path}")
    
    plt.show()
else:
    print("ERROR: No strategies to evaluate")
    coherence_results = {}


# SECTION 9: RETRIEVAL QUALITY TEST

Real-world test: Retrieval Quality
-----------------------------------
Test dengan actual queries dan evaluate apakah correct chunks retrieved

In [ ]:
print("\n" + "="*80)
print("RETRIEVAL QUALITY TEST")
print("="*80 + "\n")

# Test queries
test_queries = [
    "Apa saja persyaratan untuk mengajukan pengaduan?",
    "Berapa lama waktu investigasi pengaduan?",
    "Bagaimana cara menyampaikan pengaduan secara online?",
    "Apa yang dimaksud dengan maladministrasi?"
]

# Expected relevant content (keywords untuk matching)
expected_content_keywords = {
    test_queries[0]: ["persyaratan", "administratif", "identitas", "pelapor"],
    test_queries[1]: ["60 hari", "investigasi", "diselesaikan", "jangka waktu"],
    test_queries[2]: ["surat elektronik", "aplikasi", "daring", "online", "disampaikan"],
    test_queries[3]: ["maladministrasi", "perilaku", "melawan hukum", "definisi"]
}

def evaluate_retrieval_fixed(query: str, chunks: List[Chunk], model, 
                              expected_keywords: List[str], top_k: int = 3):
    """
    Retrieve top-k chunks untuk query dan evaluate relevance
    """
    if not chunks:
        return []
    
    # Encode query
    query_embedding = model.encode([query], show_progress_bar=False)
    
    # Encode chunks
    chunk_texts = [c.text for c in chunks]
    chunk_embeddings = model.encode(chunk_texts, show_progress_bar=False)
    
    # Calculate similarity
    similarities = cosine_similarity(query_embedding, chunk_embeddings)[0]
    
    # Get top-k
    top_indices = np.argsort(similarities)[::-1][:top_k]
    
    results = []
    for idx in top_indices:
        chunk = chunks[idx]
        similarity = similarities[idx]
        
        # Check relevance: apakah chunk mengandung expected keywords
        chunk_lower = chunk.text.lower()
        keyword_matches = sum(1 for kw in expected_keywords if kw.lower() in chunk_lower)
        is_relevant = keyword_matches >= 2  # At least 2 keywords match
        
        results.append({
            'chunk': chunk,
            'similarity': similarity,
            'chunk_id': chunk.chunk_id,
            'is_relevant': is_relevant,
            'keyword_matches': keyword_matches
        })
    
    return results

# Run retrieval test untuk each strategy
if strategies_stats:
    print("Running retrieval tests with fixed evaluation...\n")
    
    retrieval_results_fixed = {}
    
    for strategy, stats in strategies_stats.items():
        print(f"\nTesting: {strategy}")
        print("-"*80)
        
        strategy_results = []
        
        for query in test_queries:
            expected_kw = expected_content_keywords[query]
            results = evaluate_retrieval_fixed(query, stats['chunks'], model, 
                                              expected_kw, top_k=3)
            
            if results:
                print(f"\nQuery: {query}")
                print(f"Top 3 retrieved chunks:")
                for i, r in enumerate(results, 1):
                    relevance_marker = "[RELEVANT]" if r['is_relevant'] else ""
                    print(f"  {i}. {r['chunk_id']} (sim: {r['similarity']:.4f}, kw_match: {r['keyword_matches']}) {relevance_marker}")
                    print(f"     Preview: {r['chunk'].text[:100]}...")
                
                # Check if any top-3 is relevant
                hit = any(r['is_relevant'] for r in results)
                top_relevant = results[0]['is_relevant']
                
                strategy_results.append({
                    'query': query,
                    'top_similarity': results[0]['similarity'],
                    'hit': hit,
                    'top_relevant': top_relevant,
                    'top_keyword_matches': results[0]['keyword_matches']
                })
            else:
                print(f"\nQuery: {query}")
                print("  No results (empty chunks)")
        
        retrieval_results_fixed[strategy] = strategy_results
    
    # Calculate metrics
    print("\n" + "="*80)
    print("RETRIEVAL ACCURACY SUMMARY")
    print("="*80 + "\n")
    
    accuracy_data_fixed = []
    for strategy, results in retrieval_results_fixed.items():
        if results:
            # Hit@3: Any of top-3 is relevant
            hits = sum(1 for r in results if r['hit'])
            accuracy_top3 = hits / len(results) * 100
            
            # Precision@1: Top-1 is relevant
            top_relevant_count = sum(1 for r in results if r['top_relevant'])
            precision_at_1 = top_relevant_count / len(results) * 100
            
            avg_similarity = np.mean([r['top_similarity'] for r in results])
            avg_kw_matches = np.mean([r['top_keyword_matches'] for r in results])
            
            accuracy_data_fixed.append({
                'Strategy': strategy,
                'Hit@3 (%)': f"{accuracy_top3:.1f}",
                'Precision@1 (%)': f"{precision_at_1:.1f}",
                'Avg Top Similarity': f"{avg_similarity:.4f}",
                'Avg Keyword Matches': f"{avg_kw_matches:.1f}"
            })
        else:
            accuracy_data_fixed.append({
                'Strategy': strategy,
                'Hit@3 (%)': "0.0",
                'Precision@1 (%)': "0.0",
                'Avg Top Similarity': "0.0000",
                'Avg Keyword Matches': "0.0"
            })
    
    accuracy_df_fixed = pd.DataFrame(accuracy_data_fixed)
    print(accuracy_df_fixed.to_string(index=False))
    
    # Save to CSV
    accuracy_csv_path_fixed = 'output_tables/retrieval_accuracy_fixed.csv'
    accuracy_df_fixed.to_csv(accuracy_csv_path_fixed, index=False)
    print(f"\nTable saved to: {accuracy_csv_path_fixed}")
    
    # Visualization - Compare Hit@3 and Precision@1
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    strategies_list = [d['Strategy'] for d in accuracy_data_fixed]
    hit_at_3 = [float(d['Hit@3 (%)']) for d in accuracy_data_fixed]
    precision_at_1 = [float(d['Precision@1 (%)']) for d in accuracy_data_fixed]
    colors = ['#3498db', '#e74c3c', '#2ecc71']
    
    # Plot 1: Hit@3
    ax1 = axes[0]
    bars1 = ax1.bar(strategies_list, hit_at_3, color=colors, alpha=0.7, edgecolor='black')
    ax1.set_ylabel('Hit@3 Accuracy (%)', fontweight='bold')
    ax1.set_title('Hit@3: Relevant Chunk in Top-3 Results', fontweight='bold')
    ax1.set_ylim(0, 110)
    ax1.grid(axis='y', alpha=0.3)
    ax1.axhline(y=100, color='green', linestyle='--', alpha=0.5, linewidth=1)
    
    for bar, val in zip(bars1, hit_at_3):
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height + 2,
                f'{val:.1f}%',
                ha='center', va='bottom', fontweight='bold', fontsize=11)
    
    # Plot 2: Precision@1
    ax2 = axes[1]
    bars2 = ax2.bar(strategies_list, precision_at_1, color=colors, alpha=0.7, edgecolor='black')
    ax2.set_ylabel('Precision@1 (%)', fontweight='bold')
    ax2.set_title('Precision@1: Top-1 Result is Relevant', fontweight='bold')
    ax2.set_ylim(0, 110)
    ax2.grid(axis='y', alpha=0.3)
    ax2.axhline(y=100, color='green', linestyle='--', alpha=0.5, linewidth=1)
    
    for bar, val in zip(bars2, precision_at_1):
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height + 2,
                f'{val:.1f}%',
                ha='center', va='bottom', fontweight='bold', fontsize=11)
    
    plt.tight_layout()
    
    # Save figure
    accuracy_fig_path_fixed = 'output_figures/retrieval_accuracy_fixed.png'
    plt.savefig(accuracy_fig_path_fixed, dpi=300, bbox_inches='tight')
    print(f"Figure saved to: {accuracy_fig_path_fixed}")
    
    plt.show()
    
    print("\n" + "="*80)
    print("KEY FINDINGS (CORRECTED):")
    print("="*80)
    
    # Identify best strategy
    if hit_at_3:
        best_hit3_idx = hit_at_3.index(max(hit_at_3))
        best_precision_idx = precision_at_1.index(max(precision_at_1))
        
        print(f"\nBest Hit@3: {strategies_list[best_hit3_idx]} ({hit_at_3[best_hit3_idx]:.1f}%)")
        print(f"Best Precision@1: {strategies_list[best_precision_idx]} ({precision_at_1[best_precision_idx]:.1f}%)")
        
        print("\nINTERPRETATION:")
        print("- Hit@3: Measures if relevant content appears in top-3 results")
        print("- Precision@1: Measures if THE BEST result is actually relevant")
        print("\nHigher scores = better chunking strategy for RAG retrieval")
else:
    print("ERROR: No strategies to test retrieval")


# SECTION 10: SUMMARY & NEXT STEPS

In [ ]:
print("\n\n" + "="*80)
print("SUMMARY - APA YANG SUDAH KITA PELAJARI")
print("="*80 + "\n")

summary_points = [
    "EMBEDDING: Text dikonversi menjadi vector representation",
    "SEMANTIC SIMILARITY: Mengukur kesamaan makna, bukan kata",
    "SEMANTIC SEARCH: Lebih powerful dari keyword search",
    "XLM-RoBERTa: Model yang support Bahasa Indonesia dengan baik",
    "RAG FOUNDATION: Retrieval based on semantic understanding"
]

for point in summary_points:
    print(f"  {point}")

print("\n" + "="*80)
print("ACTIONABLE WORKFLOW UNTUK PRODUCTION")
print("="*80 + "\n")

workflow = """
RECOMMENDATION FOR OMBUDSMAN:

Use LEGAL DOCUMENT-AWARE CHUNKING (Pasal-level) karena:
  1. Preserves semantic units (1 Pasal = 1 konsep legal)
  2. Maintains cross-references between Pasal
  3. Easier untuk maintenance (update 1 Pasal = 1 chunk)
  4. Aligns dengan mental model staff

PRODUCTION CONSIDERATIONS:
  - Peraturan/UU: Chunk by Pasal
  - SOP: Chunk by procedure steps
  - General docs: Sentence-based as fallback
  - Optimal chunk size: 100-300 words (based on XLM-RoBERTa context)
"""

print(workflow)

print("\n" + "="*80)
print("NEXT: AFTERNOON SESSION - PRODUCTION IMPLEMENTATION")
print("="*80 + "\n")

print("After BREAK (13.00):")
print("  - Build complete RAG pipeline dengan optimal chunking")
print("  - Apply pada YOUR documents (local MacBook)")
print("  - Integrate: Document Processing + Chunking + Embedding + Retrieval")
print("  - Deployment considerations untuk Komdigi infrastructure")
print()

print("Notebook 03 COMPLETE!")
print("\nAll outputs saved:")
print(f"  - Tables: output_tables/")
print(f"  - Figures: output_figures/")
print()